# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [2]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [3]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

print(len(metadata_new)) # 6053 rows

7397
6053


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\4\ipykernel_17196\2731231253.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


In [4]:
# Modify isolates to xxxxxx-xxx

# new_isolates = []
# for isolate in metadata_new["isolate"]:
#     new_isolate = ""
#     # if isolate:
#     #     print(isolate)
#     try:
#         split_isolate = isolate.split("-")
#     except:
#         new_isolate = ""
#     # else:
#     #     split_isolate = ""
#     for split in split_isolate:
#         if len(split) == 6:
#             new_isolate = new_isolate + split
#         if len(split) == 3:
#             new_isolate = new_isolate + "-" + split
#     new_isolates.append(new_isolate)

# metadata_new["isolate"] = new_isolates

# display(metadata_new["isolate"])

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use output.tsv

In [5]:
# Get genotype from genoflu
os.chdir(temp_files)
output_tsv = pd.read_csv("output.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# print(b313_and_d11_only)
print(len(b313_and_d11_only)) # 5160 rows

metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

print(len(metadata_new)) # 5160

5160
5160


In [6]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_new, animals_ref) # Get host type
metadata_new["years"] = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [7]:
print(len(metadata_new))

5160


In [8]:
# Get geolocation from genbank_mapping.tsv

os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])
# genbank_mapping["isolate"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[3])

# Change isolate names in genbank mapping

# new_isolates = []
# for isolate in genbank_mapping["isolate"]:
#     new_isolate = ""
#     # if isolate:
#     #     print(isolate)
#     try:
#         split_isolate = isolate.split("-")
#     except:
#         continue
#     # else:
#     #     split_isolate = ""
#     for split in split_isolate:
#         if len(split) == 6:
#             new_isolate = new_isolate + split
#         if len(split) == 3:
#             new_isolate = new_isolate + "-" + split
#     new_isolates.append(new_isolate)

# genbank_mapping["isolate"] = new_isolates


# map to metadata


metadata_genbank = metadata_new.merge(genbank_mapping, on="Run", how="inner")

print(len(metadata_genbank))
display(metadata_genbank)

3357


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Host_Type,years,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,other,2024,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,cattle,2024,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,cattle,2024,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,cattle,2024,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,cattle,2024,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,2025,...,cattle,2025,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA
3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,2025,...,cattle,2025,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA
3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,2025,...,cattle,2025,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA
3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,2025,...,avian,2025,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA


In [9]:
# # Get collection date from GenBank eutils 

# def search_collection_date(biosample):

#     print(biosample)

#     try:

#         # Avoid spamming the server
#         time.sleep(2)
    
#         base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
#         search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#         # Get Biosample ID from search_url
#         output = requests.get(search_url)
#         xml = output.content
#         root = ET.fromstring(xml)
#         sample_id = root.find("./IdList/Id").text

#         biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
        
#         # Get Nucleotide ID from biosample_url
#         output = requests.get(biosample_url)
#         xml = output.content
#         root = ET.fromstring(xml)
#         query_key = root.find(".//QueryKey").text
#         web_env = root.find(".//WebEnv").text

#         nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#         output = requests.get(nucleotide_url) 
#         xml = output.content
#         root = ET.fromstring(xml)

#         # Grab collection date at the end of the sub name
#         collection_date = root.find(".//SubName").text.split("|")[-1]

#         return collection_date
    
#     except:
#         print("Unable to find collection date.")

#         if len(metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"]) > 0: # If a year exists
#             collection_date = metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"].values[0]
#         else:
#             collection_date = float('nan') 

#         return collection_date
    
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(search_collection_date)

In [10]:
# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [11]:
# Upload saved data
os.chdir(temp_files)
metadata_genbank = pd.read_csv("metadata_genbank.csv")
display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Name
0,0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,...,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas,16-Mar-2024,>A/Blackbird/Texas/24-008354-001-original/2024...
1,1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,...,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-005-original/2024|H5...
2,2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,...,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-004-original/2024|H5...
3,3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,...,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-003-original/2024|H5...
4,4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,...,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-002-original/2024|H5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,...,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-003/2025|H5N1|05-Nov-20...
3353,3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,...,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-002/2025|H5N1|05-Nov-20...
3354,3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,...,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-001/2025|H5N1|05-Nov-20...
3355,3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,...,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA,22-Jan-2025,>A/DUCK/CA/25-003063-003/2025|H5N1|22-Jan-2025...


In [12]:
# Make names

# names = ">A/" + metadata_new["Host"] + "/" + metadata_new["geo_loc_name"] + "/" + metadata_new["isolate"] + "/" + years + "|H5N1|" + metadata_new["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata_new["Host_Type"] + "|" + metadata_new["Genotype"]
names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"] + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Name
0,0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,...,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas,16-Mar-2024,>A/Blackbird/Texas/24-008354-001-original/2024...
1,1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,...,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-005-original/2024|H5...
2,2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,...,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-004-original/2024|H5...
3,3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,...,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-003-original/2024|H5...
4,4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,...,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas,20-Mar-2024,>A/Cattle/Texas/24-009108-002-original/2024|H5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,3352,SRR32653896,WGS,147.21,170346135,PRJNA1102327,SAMN47305024,Viral,58690418,USDA-NVSL,...,SRR32653896_HA_cns.fa,Consensus_SRR32653896_HA_cns_threshold_0.5_qua...,SRR32653896,HA,PV338836.1,4,A/cattle/CA/25-003262-003-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-003/2025|H5N1|05-Nov-20...
3353,3353,SRR32653897,WGS,148.06,69910137,PRJNA1102327,SAMN47305023,Viral,23973854,USDA-NVSL,...,SRR32653897_HA_cns.fa,Consensus_SRR32653897_HA_cns_threshold_0.5_qua...,SRR32653897,HA,PV338828.1,4,A/cattle/CA/25-003262-002-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-002/2025|H5N1|05-Nov-20...
3354,3354,SRR32653898,WGS,148.21,61895474,PRJNA1102327,SAMN47305022,Viral,21444614,USDA-NVSL,...,SRR32653898_HA_cns.fa,Consensus_SRR32653898_HA_cns_threshold_0.5_qua...,SRR32653898,HA,PV338820.1,4,A/cattle/CA/25-003262-001-original/2024,CA,05-Nov-2024,>A/CATTLE/CA/25-003262-001/2025|H5N1|05-Nov-20...
3355,3355,SRR32653899,WGS,148.15,67537431,PRJNA1102327,SAMN47305021,Viral,25457165,USDA-NVSL,...,SRR32653899_HA_cns.fa,Consensus_SRR32653899_HA_cns_threshold_0.5_qua...,SRR32653899,HA,PV336676.1,4,A/Duck/CA/25-003063-003-original/2025,CA,22-Jan-2025,>A/DUCK/CA/25-003063-003/2025|H5N1|22-Jan-2025...


In [13]:
# Make fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

In [24]:
# print(fasta_files[list(fasta_files.keys())[0]])

print(len(fasta_files["D1.1_PB2"]))

# print(fasta_files["B3.13_PB1"])

602


In [15]:
# Create fasta files 
os.chdir(complete_files)
for pair in fasta_files.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/Blackbird/Texas/24-008354-001-original/2024|H5N1|16-Mar-2024|other|B3.13
>A/Cattle/Texas/24-009108-005-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009108-004-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009108-003-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009108-002-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009108-001-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009088-002-original/2024|H5N1|13-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009088-001-original/2024|H5N1|13-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009087-001-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009029-001-original/2024|H5N1|21-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009028-019-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Chicken/Texas/24-007264-003-original/2024|H5N1|07-Mar-2024|avian|B3.13
>A/Cattle/Texas/24-009028-009-original/2024|H5N1|20-Mar-2024|cattle|B3.13
>A/Cattle/Texas/24-009028-008-origin

In [174]:
# De-duplication attempt 2

from collections import defaultdict

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/"

os.chdir(gisaid)

dfs_gisaid = defaultdict(list)
for dirpath, dirs, files in os.walk(gisaid): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        gisaid_df = pd.DataFrame()
        with open(file_name) as f:
            lines = f.readlines()
            isolate_partial = []
            full_header = []
            sequence = []
            # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
            for num, line in enumerate(lines):
                if line[0] == ">": # If it's a header
                    full_header.append(line)
                    full = line.split("/")[3] # Get the isolate
                    partial = full.split("_")[-1] # If 25_, get the last bit
                    digits = partial.split("-")
                    isolate = ""
                    other = ""
                    for d in digits:
                        # print(d)
                        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                            isolate = d + "-"
                        elif len(d) == 3 and d.isnumeric():
                            isolate = isolate + d
                        elif d.isnumeric() == False: # If it's a weird isolate
                            other = d + "-"
                        else: 
                            other = other + d
                    # Now add to list to check in Andersen files without doing wild for loops
                    if len(isolate) == 10: # If this is a correctly formatted isolate
                        # isolates.append(isolate)
                        # All headers are followed by sequences
                        isolate_partial.append(isolate)
                    else: # If this is some other isolate
                        isolate_partial.append(other)
                else: # It's a sequence
                    sequence.append(line)
            gisaid_df["isolate_partial"] = isolate_partial
            gisaid_df["full_header"] = full_header
            gisaid_df["sequence"] = sequence
            dfs_gisaid[file_name.split("/")[-1][:-6]].append(gisaid_df)

In [175]:
print(dfs_gisaid["B3.13_HA"][0])

       isolate_partial                                        full_header  \
0           038428-002  >A/chicken/USA/038428-002/2024|H5N1|2024|avian...   
1           035208-001  >A/dairy_cow/USA/035208-001/2024|H5N1|2024|cat...   
2           003574-003  >A/turkey/California/003574-003/2025|H5N1|2025...   
3           038428-001  >A/chicken/USA/038428-001/2024|H5N1|2024|avian...   
4           003903-001  >A/turkey/California/003903-001/2025|H5N1|2025...   
..                 ...                                                ...   
778         005913-003  >A/cat/USA/005913-003/2025|H5N1|2025|feline|B3...   
779         004757-001  >A/environment/USA/004757-001/2025|H5N1|2025|o...   
780         004757-002  >A/environment/USA/004757-002/2025|H5N1|2025|o...   
781         034828-002  >A/dairy_cow/California/24_034828-002-R2/2024|...   
782  10|human|B3.13\n-  >A/California/216/2024|H5N1|2024-12-10|human|B...   

                                              sequence  
0    atgaagaacatag

In [176]:
# Do the same with Andersen 

dfs_andersen = defaultdict(list)
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        andersen_df = pd.DataFrame()
        with open(file_name) as f:
            lines = f.readlines()
            isolate_partial = []
            full_header = []
            sequence = []
            # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
            for num, line in enumerate(lines):
                if line[0] == ">": # If it's a header
                    full_header.append(line)
                    full = line.split("/")[3] # Get the isolate
                    partial = full.split("_")[-1] # If 25_, get the last bit
                    digits = partial.split("-")
                    isolate = ""
                    other = ""
                    for d in digits:
                        # print(d)
                        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                            isolate = d + "-"
                        elif len(d) == 3 and d.isnumeric() and len(isolate) < 10: # If we haven't already gotten the isolate
                            isolate = isolate + d
                        elif d.isnumeric() == False: # If it's a weird isolate
                            other = d + "-"
                        else: 
                            other = other + d
                    # Now add to list to check in Andersen files without doing wild for loops
                    if len(isolate) == 10: # If this is a correctly formatted isolate
                        # print(isolate)
                        # isolates.append(isolate)
                        # All headers are followed by sequences
                        isolate_partial.append(isolate)
                    else: # If this is some other isolate
                        # print(full)
                        # print(other)
                        isolate_partial.append(other)
                else: # It's a sequence
                    sequence.append(line)
            andersen_df["isolate_partial"] = isolate_partial
            andersen_df["full_header"] = full_header
            andersen_df["sequence"] = sequence
            dfs_andersen[file_name.split("/")[-1][:-6]].append(andersen_df)

In [180]:
print(dfs_andersen["B3.13_HA"][0])

     isolate_partial                                        full_header  \
0         000003-001  >A/CHICKEN/CA/25-000003-001/2025|H5N1|28-Dec-2...   
1         000003-001  >A/chicken/California/25-000003-001/2024|H5N1|...   
2         000003-001  >A/chicken/USA/000003-001/2025|H5N1|2025|avian...   
3         000026-001  >A/dairy_cow/USA/000026-001/2025|H5N1|2025|cat...   
4         000026-002  >A/dairy_cow/USA/000026-002/2025|H5N1|2025|cat...   
...              ...                                                ...   
3472   A241900097-37  >A/dairy_cow/Texas/A241900097-37/2024|H5N1|202...   
3473           F001-  >A/cat/New_Mexico/F001/2024|H5N1|2024|feline|B...   
3474            SM-3  >A/dairy_cow/Kansas/SM-3/2024|H5N1|2024-04-12|...   
3475            SM-6  >A/dairy_cow/Kansas/SM-6/2024|H5N1|2024-04-12|...   
3476       T2402448-  >A/dairy_cow/California/T2402448/2024|H5N1|202...   

                                               sequence  
0     ATGAAGAACATAGTACTACTTCTTGCAATAGTTAG

In [183]:
full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        full_df = df.merge(dfs_gisaid[key][i], how="outer")
        # print(full_df)
        full_df = full_df.drop_duplicates(subset=["isolate_partial"])
        full_dfs[key].append(full_df)



In [184]:
print(full_dfs["B3.13_HA"][0])

     isolate_partial                                        full_header  \
0         000003-001  >A/CHICKEN/CA/25-000003-001/2025|H5N1|28-Dec-2...   
3         000026-001  >A/dairy_cow/USA/000026-001/2025|H5N1|2025|cat...   
4         000026-002  >A/dairy_cow/USA/000026-002/2025|H5N1|2025|cat...   
5         000045-001  >A/CHICKEN/CA/25-000045-001/2025|H5N1|29-Dec-2...   
7         000046-001  >A/chicken/USA/000046-001/2025|H5N1|2025|avian...   
...              ...                                                ...   
3472   A241900097-37  >A/dairy_cow/Texas/A241900097-37/2024|H5N1|202...   
3473           F001-  >A/cat/New_Mexico/F001/2024|H5N1|2024|feline|B...   
3474            SM-3  >A/dairy_cow/Kansas/SM-3/2024|H5N1|2024-04-12|...   
3475            SM-6  >A/dairy_cow/Kansas/SM-6/2024|H5N1|2024-04-12|...   
3476       T2402448-  >A/dairy_cow/California/T2402448/2024|H5N1|202...   

                                               sequence  
0     ATGAAGAACATAGTACTACTTCTTGCAATAGTTAG

In [185]:
# Create fasta files 
os.chdir(complete_files)
for pair in full_dfs.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
        # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

In [ ]:
# De-duplication with GISAID

gisaid = downloads + "GISAID_Complete_Fasta_Files/"

os.chdir(gisaid)

isolates = {}
for dirpath, dirs, files in os.walk(gisaid): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        to_append = []
        with open(file_name) as f:
            lines = f.readlines()
            # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
            for num, line in enumerate(lines):
                if line[0] == ">": # If it's a header
                    full = line.split("/")[3] # Get the isolate
                    partial = full.split("_")[-1] # If 25_, get the last bit
                    digits = partial.split("-")
                    isolate = ""
                    other = ""
                    for d in digits:
                        # print(d)
                        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                            isolate = d + "-"
                        elif len(d) == 3 and d.isnumeric():
                            isolate = isolate + d
                        elif d.isnumeric() == False: # If it's a weird isolate
                            other = d + "-"
                        else: 
                            other = other + d
                    # Now add to list to check in Andersen files without doing wild for loops
                    if len(isolate) == 10: # If this is a correctly formatted isolate
                        # isolates.append(isolate)
                        # All headers are followed by sequences
                        to_append.append([isolate, line, lines[num + 1]])
                    else: # If this is some other isolate
                        to_append.append([other, line, lines[num + 1]])
        isolates[file_name.split("/")[-1][:-6]] = to_append

print(len(isolates["D1.1_PB2"]))



1615


In [125]:
# If this isolate is not already in the Andersen fasta files, add it to those files

os.chdir(complete_files)


print(len(metadata_genbank))
# Some names don't exist
metadata_genbank = metadata_genbank.dropna(subset="Name")
print(len(metadata_genbank))
# Exclude duplicates
metadata_genbank = metadata_genbank.drop_duplicates(subset="Name")
print(len(metadata_genbank))

andersen_isolates = {}
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        to_append = []
        with open(file_name) as f:
            lines = f.readlines()
            # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
            for num, line in enumerate(lines):
                if line[0] == ">": # If it's a header
                    print(num)

                    full = line.split("/")[3] # Get the isolate
                    partial = full.split("_")[-1] # If 25_, get the last bit
                    digits = partial.split("-")
                    isolate = ""
                    other = ""
                    for d in digits:
                        # print(d)
                        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                            isolate = d + "-"
                        elif len(d) == 3 and d.isnumeric():
                            isolate = isolate + d
                        elif d.isnumeric() == False: # If it's a weird isolate
                            other = d + "-"
                        else: 
                            other = other + d
                    # Now add to list to check in Andersen files without doing wild for loops
                    if len(isolate) == 10: # If this is a correctly formatted isolate
                        # isolates.append(isolate)
                        # All headers are followed by sequences
                        to_append.append([isolate, line, lines[num + 1]])
                    else: # If this is some other isolate
                        to_append.append([other, line, lines[num + 1]])
        andersen_isolates[file_name.split("/")[-1][:-6]] = to_append



# print(to_remove_from_gisaid)

print((andersen_isolates["D1.1_PB2"][0]))
print(len(andersen_isolates["D1.1_PB2"]))

# for entry in to_remove_from_gisaid:
#     isolates.pop(entry)
# print(isolates)
# Now add remaining GISAID data to Andersen data
# isolates dictionary format: {isolate : [name, sequence]}

# gisaid_data = pd.DataFrame()

# display(metadata_genbank)

# Move the GISAID sequences to Andersen FASTA files

3296
3296
3296
0
2
4
6
8
10
12
14
16
18
20
22
24
26
28
30
32
34
36
38
40
42
44
46
48
50
52
54
56
58
60
62
64
66
68
70
72
74
76
78
80
82
84
86
88
90
92
94
96
98
100
102
104
106
108
110
112
114
116
118
120
122
124
126
128
130
132
134
136
138
140
142
144
146
148
150
152
154
156
158
160
162
164
166
168
170
172
174
176
178
180
182
184
186
188
190
192
194
196
198
200
202
204
206
208
210
212
214
216
218
220
222
224
226
228
230
232
234
236
238
240
242
244
246
248
250
252
254
256
258
260
262
264
266
268
270
272
274
276
278
280
282
284
286
288
290
292
294
296
298
300
302
304
306
308
310
312
314
316
318
320
322
324
326
328
330
332
334
336
338
340
342
344
346
348
350
352
354
356
358
360
362
364
366
368
370
372
374
376
378
380
382
384
386
388
390
392
394
396
398
400
402
404
406
408
410
412
414
416
418
420
422
424
426
428
430
432
434
436
438
440
442
444
446
448
450
452
454
456
458
460
462
464
466
468
470
472
474
476
478
480
482
484
486
488
490
492
494
496
498
500
502
504
506
508
510
512
514
516
518


In [126]:
# Check if the isolate (or other) is in the keys of the dictionary made above
to_check = []
to_remove_from_gisaid = []

for key in isolates.keys():
    for isolate in isolates[key]:
        to_check.append(isolate[1])

to_check2 = []
for key in andersen_isolates.keys():
    for isolate in andersen_isolates[key]:
        to_check2.append(isolate[0])
        # print(isolate[0])

to_check = list(set(to_check))
to_check2 = list(set(to_check2))

print(len(to_check))
print(len(to_check2))

print(to_check2[100])

for c in to_check:
    for cc in to_check2:
        if cc in c: # and c not in to_remove_from_gisaid:
            to_remove_from_gisaid.append(c)

to_remove_from_gisaid = list(set(to_remove_from_gisaid))
# for i in andersen_isolates[andersen_isolate]:
# print(andersen_isolates[andersen_isolate][0])
    # print(isolate[1])
    # if isolate[1] == c and c not in to_remove_from_gisaid: # We've seen it -- first is the isolate number
    # # print(isolate[1])
    #     # print(c)
    #     to_remove_from_gisaid.append(c) # Second is the actual header

print((andersen_isolates["D1.1_PB2"][0][1]))
print(to_remove_from_gisaid[0])
print(len(to_remove_from_gisaid))

print(len(isolates["B3.13_HA"]))

# Remove it from the list
give_to_andersen = {}
for key in isolates.keys():
    list_of_fastas = isolates[key]
    # print(list_of_fastas)
    # break
    for index, fasta in enumerate(list_of_fastas):
        # print(fasta)
        # break
        for num, header in enumerate(to_remove_from_gisaid):
            if fasta[1] == header and fasta[1] == fasta[1]: # If it's in the to_remove list and not nan
                # print(num, " remove: ", header)
                try:
                    del list_of_fastas[index]
                except:
                    "Cannot remove something that doesn't exist."
    give_to_andersen[key] = list_of_fastas
    # print(len(list_of_fastas))

    # print(list_of_fastas[0])

# print(give_to_andersen["B3.13_HA"])
print(len(give_to_andersen["B3.13_HA"]))

2398
3041
017584-009
>A/Chicken/WA/24-032809-002/2024|H5N1|05-Nov-2024|avian|D1.1

>A/turkey/Wisconsin/038658-001/2024|H5N1|2024-12-24|avian|D1.1

834
783
602


In [127]:
print(len(give_to_andersen["B3.13_HA"]))
print(len(isolates["B3.13_HA"]))
print(len(to_remove_from_gisaid))
print(len(andersen_isolates["B3.13_HA"]))

602
602
834
2694


In [128]:
print(len(fasta_files["B3.13_HA"]))

2755


In [129]:
for key in give_to_andersen.keys():
    for value in give_to_andersen[key]:
        andersen_isolates[key].append(value)

print(len(andersen_isolates["B3.13_HA"]))

3296


In [134]:
# Create fasta files 
os.chdir(complete_files)
for pair in andersen_isolates.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in andersen_isolates[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[1].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[1])
        # print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name)
        output_file.write(item[2])
    output_file.close()

In [ ]:
# def search_collection_date(biosample):
    
#     base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
#     search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#     # Get Biosample ID from search_url
#     output = requests.get(search_url)
#     xml = output.content
#     root = ET.fromstring(xml)
#     sample_id = root.find("./IdList/Id").text

#     biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
    
#     # Get Nucleotide ID from biosample_url
#     output = requests.get(biosample_url)
#     xml = output.content
#     root = ET.fromstring(xml)
#     query_key = root.find(".//QueryKey").text
#     web_env = root.find(".//WebEnv").text

#     nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#     output = requests.get(nucleotide_url) 
#     xml = output.content
#     root = ET.fromstring(xml)

#     # Grab collection date at the end of the sub name
#     collection_date = root.find(".//SubName").text.split("|")[-1]

#     # Avoid spamming the server
#     time.sleep(3)
    
#     return collection_date


In [ ]:
# example = search_collection_date("SAMN41019216")
# print(example)



In [ ]:

# unique_animals_all = sort_animals_anderson(metadata_new)

# # Flatten unique_animals
# every_unique_animal = []
# for animal in unique_animals_all:
#     every_unique_animal.append(animal)

# print(every_unique_animal)

# unique_animals_set = list(set(every_unique_animal))
# # animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# # animals_df["other"] = unique_animals_set # to sort

# os.chdir(downloads)

# animals_ref = pd.read_csv("animals_ref.csv")


# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer, deal with that later

# animals_df["new"] = (different_animals)

# print(animals_df)

# animals_df.to_csv("animals_ref_to_sort.csv")
